# VoMP on Gaussian-Splat USDs

This notebook shows the full VoMP workflow on a Gaussian-splat USD that is partitioned
into named `GeomSubset` segments:

1. Load the segments from the USD.
2. Predict materials (Young's modulus, Poisson's ratio, density) with VoMP, per segment.
3. Visualize the predicted materials inline.
4. Write the materials back into a USD as Kaolin physics materials.

To load USDs, VoMP provides `vomp.inference.load_gaussian_usd`, then we run it using the `Vomp.get_gaussian_usd_materials`, and finally save the materials using `vomp.inference.save_materials`.

In [1]:
import os, shutil
import numpy as np
import matplotlib.pyplot as plt

from vomp.inference import (
    Vomp,
    save_materials,
    get_gaussian_usd_segments,
)

REPO_ROOT = os.path.abspath("../..")
USD_PATH  = os.path.join(REPO_ROOT, "tools_revised2_with_some_segments.usd")
WEIGHTS   = os.path.join(REPO_ROOT, "weights")
OUT_USD   = "tools_with_vomp_materials.usd"
SEED      = 42

# set SEGMENTS = None to auto-use every non-background segment
SEGMENTS = ["scissors", "screwdriver", "drill", "stud_finder"]

[SPARSE] Backend: spconv, Attention: flash_attn
[SPARSE][CONV] spconv algo: auto
[ATTENTION] Using backend: flash_attn


## 1. Load the segments

Each `GeomSubset` on the Gaussian prim is a named segment.

In [2]:
segments = get_gaussian_usd_segments(USD_PATH)
print(f"{len(segments)} segments in {os.path.basename(USD_PATH)}:")
for name, idx in segments.items():
    print(f"  {name:14s} {len(idx):>8,d} splats")

if SEGMENTS is None:
    SEGMENTS = [s for s in segments if s != "background"]
print("\nRunning VoMP on:", SEGMENTS)

11 segments in tools_revised2_with_some_segments.usd:
  background      749,693 splats
  speaker           4,389 splats
  screwdriver       5,909 splats
  stud_finder       3,919 splats
  rope              9,386 splats
  scissors            814 splats
  bird_house        7,738 splats
  phone_mount       8,718 splats
  drill            22,979 splats
  left_boot         6,388 splats
  right_boot        7,060 splats

Running VoMP on: ['scissors', 'screwdriver', 'drill', 'stud_finder']


## 2. Predict materials with VoMP

`Vomp.get_gaussian_usd_materials` loads a segment from the USD, normalizes it, runs the splat pipeline, and returns one prediction per splat with
coordinates mapped back to the original USD world frame.

In [3]:
model = Vomp.from_checkpoint(
    config_path=os.path.join(WEIGHTS, "inference.json"),
    geometry_checkpoint_dir=os.path.join(WEIGHTS, "geometry_transformer.pt"),
    matvae_checkpoint_dir=os.path.join(WEIGHTS, "matvae.safetensors"),
    normalization_params_path=os.path.join(WEIGHTS, "normalization_params.json"),
    use_trt=False,
)

Loading Vomp model on device: cuda
✓ Detected inference config: /mnt/data/rdagli/VoMP/weights/inference.json
✓ Using direct geometry checkpoint: /mnt/data/rdagli/VoMP/weights/geometry_transformer.pt
✓ Using direct MatVAE checkpoint: /mnt/data/rdagli/VoMP/weights/matvae.safetensors


✓ Loaded geometry encoder from: /mnt/data/rdagli/VoMP/weights/geometry_transformer.pt
✓ Loaded MatVAE from: /mnt/data/rdagli/VoMP/weights/matvae.safetensors
✓ All models loaded successfully


In [4]:
results = {}
for seg in SEGMENTS:
    print(f"\n=== {seg} ===")
    results[seg] = model.get_gaussian_usd_materials(USD_PATH, segment=seg, seed=SEED)

print(f"\n{'segment':14s} {'#splats':>8s} {'YM (GPa)':>22s} {'Poisson':>18s} {'density':>18s}")
for seg, r in results.items():
    ym, pr, rho = r["youngs_modulus"], r["poisson_ratio"], r["density"]
    print(f"{seg:14s} {len(ym):>8,d} "
          f"{ym.mean()/1e9:>7.2f} [{ym.min()/1e9:.2f},{ym.max()/1e9:.2f}] "
          f"{pr.mean():>6.3f} [{pr.min():.3f},{pr.max():.3f}] "
          f"{rho.mean():>6.0f} [{rho.min():.0f},{rho.max():.0f}]")


=== scissors ===
=== Vomp: Gaussian USD Material Estimation (segment=scissors, 814 splats) ===
=== Vomp: Splat Material Estimation ===
Step 1: Extracting features...
Using custom rendering function...
Rendering 150 Gaussian splat views...
✓ Rendered 150 views
Using custom voxelization function...
Voxelized to 669 voxels (centers method)
Loading existing features from /tmp/Vomp_gusd_tools_revised2_with_some_segments_scissors/features/dinov2_vitl14_reg.npz
Loaded features:
  Voxels: 669
  Feature dimension: 1024
  Voxel index range: 0 to 63
Step 2: Running material inference...


/home/rdagli/scratch/VoMP/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/home/rdagli/scratch/VoMP/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/rdagli/scratch/VoMP/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/home/rdagli/scratch/VoMP/.venv/lib/python3.10/site-packages/spconv/pytorch/functional.py:243: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please

Running inference on 669 voxels...
Step 3: Evaluating materials at splat centers...
✓ Evaluated materials at 814 splat centers
✓ Material estimation complete!

=== screwdriver ===
=== Vomp: Gaussian USD Material Estimation (segment=screwdriver, 5,909 splats) ===
=== Vomp: Splat Material Estimation ===
Step 1: Extracting features...
Using custom rendering function...
Rendering 150 Gaussian splat views...
✓ Rendered 150 views
Using custom voxelization function...
Voxelized to 2434 voxels (centers method)
Loading existing features from /tmp/Vomp_gusd_tools_revised2_with_some_segments_screwdriver/features/dinov2_vitl14_reg.npz
Loaded features:
  Voxels: 2434
  Feature dimension: 1024
  Voxel index range: 0 to 63
Step 2: Running material inference...
Running inference on 2434 voxels...
Step 3: Evaluating materials at splat centers...
✓ Evaluated materials at 5,909 splat centers
✓ Material estimation complete!

=== drill ===
=== Vomp: Gaussian USD Material Estimation (segment=drill, 22,979 s

## 3. Visualize the materials

We use `k3d`. Each plot shows all selected segments at their USD world positions, colored by one property over a shared range. Drag to rotate, scroll to zoom.

In [ ]:
!pip install -q k3d

In [ ]:
import k3d
from scipy.spatial import cKDTree

coords = np.concatenate([np.asarray(results[s]["query_coords_world"]) for s in SEGMENTS])
coords = np.ascontiguousarray(coords, dtype=np.float32)

_sample = coords[np.random.default_rng(0).choice(len(coords), min(len(coords), 20000), replace=False)]
_nn, _ = cKDTree(coords).query(_sample, k=2)
POINT_SIZE = float(np.median(_nn[:, 1])) * 1.5

def show_property(key, label):
    vals = np.ascontiguousarray(
        np.concatenate([np.asarray(results[s][key]) for s in SEGMENTS]), dtype=np.float32
    )
    plot = k3d.plot(name=label, grid_visible=False, camera_auto_fit=True)
    plot += k3d.points(
        positions=coords,
        attribute=vals,
        color_map=k3d.colormaps.matplotlib_color_maps.Viridis,
        color_range=[float(vals.min()), float(vals.max())],
        point_size=POINT_SIZE,
        shader="flat",
        name=label,
    )
    print(f"{label}: {vals.min():.4g} .. {vals.max():.4g}")
    plot.display()

show_property("youngs_modulus", "Young's modulus (Pa)")
show_property("poisson_ratio", "Poisson's ratio")
show_property("density", "Density (kg/m^3)")


Young's modulus (Pa): 5.248e+08 .. 4.112e+09


Output()

Poisson's ratio: 0.324 .. 0.3808


Output()

Density (kg/m^3): 515.3 .. 1519


Output()

## 4. Write materials back to the USD

Copy the input USD once, then attach each segment's prediction as a **named Kaolin
physics material** (`material_name=<segment>`) on the Gaussian prim. Because predicted
coordinates were mapped back to the USD world frame, the stored points align exactly with
the original splats.

In [6]:
shutil.copyfile(USD_PATH, OUT_USD)
for seg, r in results.items():
    save_materials(r, OUT_USD, input_usd_path=OUT_USD, material_name=seg)
print("Wrote", os.path.abspath(OUT_USD))

Saved materials to: tools_with_vomp_materials.usd
Saved materials to: tools_with_vomp_materials.usd
Saved materials to: tools_with_vomp_materials.usd
Saved materials to: tools_with_vomp_materials.usd
Wrote /mnt/data/rdagli/VoMP/gradio/examples/tools_with_vomp_materials.usd


In [7]:
# list the physics materials now present on the gaussian prim.
from pxr import Usd
stage = Usd.Stage.Open(OUT_USD)
prim = stage.GetPrimAtPath(results[SEGMENTS[0]]["scene_path"])
for seg in SEGMENTS:
    has = prim.HasAPI("KaolinPhysicsMaterialAPI", seg)
    pts = prim.GetAttribute(f"kaolin_physics_material:{seg}:pts").Get()
    yms = prim.GetAttribute(f"kaolin_physics_material:{seg}:yms").Get()
    print(f"{seg:14s} api={has}  pts={len(pts) if pts else 0}  yms={len(yms) if yms else 0}")

scissors       api=True  pts=814  yms=814
screwdriver    api=True  pts=5909  yms=5909
drill          api=True  pts=22979  yms=22979
stud_finder    api=True  pts=3919  yms=3919
